# 06 — Matched control panels (Supplement S3)
Rebuilds, from the raw HEST counts, (A) the TRAIN-only gene statistics, (B) the 200 matched control panels per rule (md primary, mdv sensitivity), (C) the training-free evaluation of the paired controls, (D) re-evaluates the stored trained panel runs (expression-only: 123 runs, expression+spatial: 123 runs) from their checkpoints, and (E) the spatial-contribution comparison. Every regenerated table is compared with the stored one.
No training is performed. The panel runs were trained once in the revision (GPU training is not bit-reproducible); this notebook verifies that the stored checkpoints reproduce the stored gene-level results and that all downstream tables follow from them. Needs the raw `.h5ad` files under `hest_expression/` (A and C take several minutes).

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, json, pandas as pd, numpy as np, torch
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability, effective_mask, score_mask
from sasp_gat.evaluation import load_model, eval_both
from sasp_gat.training import evaluate_multimask_av

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
ppi_ei = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order, device)
CKDIR = REPO / 'checkpoints' if (REPO / 'checkpoints' / 'full_avail_seed42.pt').exists() else Path(BASE) / 'revision_checkpoints_2026-10-18'
print('checkpoint dir:', CKDIR)
import torch_geometric, scipy; print('versions: torch', torch.__version__, '| pyg', torch_geometric.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__)
import warnings; warnings.filterwarnings('ignore')
H5_DIR = Path(BASE) / 'hest_expression'
SP_DIR = Path(BASE) / 'revision_sasp_specificity_2026-10-20'      # stored panel analysis (S3)
MR_DIR = Path(BASE) / 'revision_masking_robustness_2026-10-19'
OUT_DIR = Path(BASE) / 'repro_outputs_2026-10-24'; OUT_DIR.mkdir(exist_ok=True)
DS = pyg_dataset; GENES = list(gene_order)
SPLIT = {s: DS[s].split for s in DS}; PLAT = {s: DS[s].platform for s in DS}
S_TR = [s for s in sorted(DS) if SPLIT[s] == 'train']
from sasp_gat import panels as pn
from sasp_gat.verify import compare_csv, report_compare


In [ ]:
# A) TRAIN-only gene statistics (normalisation X -> CP10k -> log1p is asserted against the package on every TRAIN slide)
files = pn.find_h5(H5_DIR); assert all(s in files for s in DS), 'missing h5ad files'
import time; t0 = time.time()
VNU, DUPS = pn.scan_var_names(files, sorted(DS)); print('header scan %.0f s' % (time.time() - t0))
S_VA = [s for s in sorted(DS) if SPLIT[s] == 'val']; S_TE = [s for s in sorted(DS) if SPLIT[s] == 'test']; S_HD = [s for s in sorted(DS) if SPLIT[s] == 'external_hd']
UNI = {k: pn.intersect(VNU, v) for k, v in dict(tr=S_TR, va=S_VA, te=S_TE, hd=S_HD).items()}
G, errs = pn.train_gene_stats(files, DS, GENES, S_TR, VNU)
print('max |package - recomputed| over TRAIN slides: %.2e' % max(errs.values()))
S = G.loc[GENES]; POOL = G[(G.n_slides == len(S_TR)) & (~G.index.isin(GENES))]
ELIG = UNI['tr'] & UNI['va'] & UNI['te'] & UNI['hd']; POOL_E = POOL.loc[POOL.index.isin(ELIG)]
print('pool genes', len(POOL), '| eligible in all cohorts', len(POOL_E))
GS = G.assign(is_sasp=G.index.isin(GENES), in_pool=G.index.isin(POOL.index), eligible_all_cohorts=G.index.isin(ELIG)); GS.index.name = 'gene'
GS_out = GS.reset_index(); GS_out.to_csv(OUT_DIR / 'train_gene_stats.csv', index=False)
report_compare({'train_gene_stats.csv': compare_csv(GS_out, SP_DIR / 'train_gene_stats.csv', rtol=1e-6, atol=1e-9)})

In [ ]:
# B) matched panels (md: mean+detection, primary; mdv: +variance, sensitivity); thresholds fixed beforehand and not amended
PAN, REP = pn.matched_panels(S, POOL_E, GENES)
for r, q in REP.items(): print(r, '| A2 (panel-level) pass:', q['a2'], '| A4 (pair-level) pass:', q['a4'], '| panels failing pair-level thresholds:', int((~q['flag']).sum()), '/ 200')
PAN.to_csv(OUT_DIR / 'panels_matched_v1.csv', index=False)
report_compare({'panels_matched_v1.csv': compare_csv(PAN, SP_DIR / 'panels_matched_v1.csv', rtol=1e-9, atol=1e-12)})

In [ ]:
# C) bank of panel genes + training-free evaluation of SASP and paired controls (Baseline A, NeighborMean, Ring2Mean)
BANK, UG = pn.build_bank(files, DS, SPLIT, PLAT, GENES, PAN, DUPS)
f_res = OUT_DIR / 'res17_per_slide_mask_seed.csv'; f_res = f_res if f_res.exists() else MR_DIR / 'res17_per_slide_mask_seed.csv'
RES17 = pd.read_csv(f_res)
from sasp_gat.masking import make_mask
_s = next(iter(test_s)); _n = test_s[_s].x.shape[0]
assert torch.equal(make_mask(_n, mask_ratio=0.25, deterministic_seed_str=f'{_s}_test_0'), pn.make_mask_local(_n, f'{_s}_test_0')); print('mask generator identical to the protocol')
GL, PSF = pn.training_free_eval(BANK, UG, PAN, DS, SPLIT, G, GENES, device, ref_res17=RES17)
print('gate: SASP columns reproduce the scenario-run records (slide x mask x method)')
GL.to_csv(OUT_DIR / 'training_free_gene_level_v1.csv', index=False); PSF.to_csv(OUT_DIR / 'training_free_panel_stats_v1.csv', index=False)
report_compare({'training_free_gene_level_v1.csv': compare_csv(GL, SP_DIR / 'training_free_gene_level_v1.csv', rtol=1e-9, atol=1e-12),
                'training_free_panel_stats_v1.csv': compare_csv(PSF, SP_DIR / 'training_free_panel_stats_v1.csv', rtol=1e-9, atol=1e-12)})

In [ ]:
# D) re-evaluate the stored trained runs (checkpoints) -> gene-level tables; compare with the stored ones
SEEDS3 = [42, 123, 2026]
pm = PAN.groupby(['rule', 'panel']).panel_a4_pass.agg(['first', 'nunique']).reset_index(); assert (pm['nunique'] == 1).all()
MD_PANELS = [int(x) for x in sorted(pm[pm.rule == 'md'].panel)[:30]]
MDV_PANELS = [int(x) for x in sorted(pm[(pm.rule == 'mdv') & pm['first'].astype(bool)].panel)[:10]]
GROUPS = [('sasp', 'sasp', -1)] + [('ctrl', 'md', p) for p in MD_PANELS] + [('ctrl', 'mdv', p) for p in MDV_PANELS]
R2REF = pn.ring2_reference(GL); TRAIN_VAR = G['var'].astype(float).to_dict()
ARMS = {'expression+spatial': dict(dir=SP_DIR / 'trained_runs_v1', prefix='', spatial=True, stored='trained_gene_level_v1.csv'),
        'expression-only': dict(dir=SP_DIR / 'trained_runs_expronly_v1', prefix='exprop_', spatial=False, stored='exprop_gene_level_v1.csv')}
GENE_TABLES = {}
for arm, cfg in ARMS.items():
    rows = []; t0 = time.time()
    for kind, rule, panel in GROUPS:
        PK = pn.make_packs(kind, rule, panel, PAN, GENES, UG, BANK, DS, SPLIT, device)
        genes_run = GENES if kind == 'sasp' else PAN[(PAN.rule == rule) & (PAN.panel == panel)].control_gene.tolist()
        for seed in SEEDS3:
            rid = f"{cfg['prefix']}sasp_s{seed}" if kind == 'sasp' else f"{cfg['prefix']}ctrl_{rule}_p{panel}_s{seed}"
            res, ck = pn.reeval_run(cfg['dir'] / f'{rid}.pt', cfg['spatial'], PK, device)
            assert (ck['kind'], ck['rule'], int(ck['panel']), int(ck['seed'])) == (kind, rule, int(panel), seed)
            rows += pn.gene_rows(kind, rule, panel, seed, res, genes_run, R2REF, TRAIN_VAR)
        del PK
    GENE_TABLES[arm] = pd.DataFrame(rows); print(arm, len(rows), 'rows,', '%.0f s' % (time.time() - t0))
    GENE_TABLES[arm].to_csv(OUT_DIR / cfg['stored'], index=False)
    # GPU float32 reductions are not bit-reproducible. Observed (Tesla T4): relative diff of ss/mse/NMSE <= 5.5e-8; RI is a ratio and can
    # move by up to ~2e-5 (absolute) where the Ring2Mean denominator is tiny (control genes only) -> rtol 1e-5, atol 1e-4
    report_compare({cfg['stored']: compare_csv(GENE_TABLES[arm], SP_DIR / cfg['stored'], rtol=1e-5, atol=1e-4)})

In [ ]:
# E) spatial contribution (expression+spatial vs expression-only), from the regenerated and from the stored gene-level tables
gs_stored = pd.read_csv(SP_DIR / 'train_gene_stats.csv')
for label, (a, b) in {'regenerated': (GENE_TABLES['expression+spatial'], GENE_TABLES['expression-only']),
                      'stored': (pd.read_csv(SP_DIR / 'trained_gene_level_v1.csv', float_precision='round_trip'), pd.read_csv(SP_DIR / 'exprop_gene_level_v1.csv', float_precision='round_trip'))}.items():
    ps, gl = pn.spatial_contribution(a, b, gs_stored)
    print(label, ps.shape, gl.shape)
    res = {f: compare_csv(df, SP_DIR / f, rtol=1e-5 if label == 'regenerated' else 1e-9, atol=1e-9) if (SP_DIR / f).exists() else (False, float('nan'), 'stored file not found')
           for f, df in {'spatial_contribution_panel_stats_v1.csv': ps, 'spatial_contribution_sasp_gene_level_v1.csv': gl}.items()}
    print('--', label); report_compare(res)
    if label == 'regenerated':
        ps.to_csv(OUT_DIR / 'spatial_contribution_panel_stats_v1.csv', index=False); gl.to_csv(OUT_DIR / 'spatial_contribution_sasp_gene_level_v1.csv', index=False)